# Mini Dashboards: Understanding Input Types and Callbacks

## Overview

This notebook contains **10 mini-dashboards**, each focusing on a **single input type** and how it controls one or more graphs through callbacks.

### What You'll Learn:
1. How different input components work
2. How to create callbacks that respond to user input
3. How to update graphs based on user selections
4. Best practices for each input type

### Input Types Covered:
1. **Dropdown** - Select one option from a list
2. **Slider** - Select a numeric value from a range
3. **Range Slider** - Select a range of values
4. **Radio Items** - Select one option (mutually exclusive)
5. **Checklist** - Select multiple options
6. **Input Text** - Enter text that filters data
7. **Date Picker** - Select a single date
8. **Date Range Picker** - Select a date range
9. **Button** - Trigger actions on click
10. **Tabs** - Switch between different views

---

## Setup

In [ ]:
!pip install dash plotly pandas dash-bootstrap-components

In [ ]:
# Import required libraries
from dash import Dash, html, dcc, Input, Output, State, callback
import dash_bootstrap_components as dbc
from dash.exceptions import PreventUpdate
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Create Sample Dataset

In [ ]:
# Create a comprehensive dataset for all mini-dashboards
np.random.seed(42)

# Generate dates
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

# Define categories
products = ['Laptop', 'Monitor', 'Keyboard', 'Mouse', 'Headphones']
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Accessories']

# Generate dataset
data = []
for _ in range(500):
    product = np.random.choice(products)
    region = np.random.choice(regions)
    category = 'Electronics' if product in ['Laptop', 'Monitor'] else 'Accessories'
    date = np.random.choice(dates)
    sales = np.random.randint(100, 5000)

    data.append({
        'Date': date,
        'Product': product,
        'Region': region,
        'Category': category,
        'Sales': sales,
        'Quantity': np.random.randint(1, 50)
    })

df = pd.DataFrame(data)
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['Year'] = df['Date'].dt.year

print(f"Dataset created with {len(df)} records")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Dataset created with 500 records
Date range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00

Columns: ['Date', 'Product', 'Region', 'Category', 'Sales', 'Quantity', 'Month', 'Year']

First few rows:


,Date,Product,Region,Category,Sales,Quantity,Month,Year
0,2023-09-28,Mouse,North,Accessories,3872,21,2023-09,2023
1,2023-08-03,Monitor,East,Electronics,4526,11,2023-08,2023
2,2024-10-25,Headphones,West,Accessories,230,22,2024-10,2024
3,2023-12-10,Headphones,South,Accessories,2533,21,2023-12,2023
4,2023-11-10,Laptop,West,Electronics,4217,44,2023-11,2023


---

# Mini Dashboard 1: Dropdown Input

## Concept:
A **dropdown** allows users to select **one option** from a list. When they select an option, it updates the graph.

## Use Cases:
- Filter by product
- Filter by region
- Select a metric to display

## Callback Flow:
```
User selects from dropdown → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 1: Dropdown
app1 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app1.layout = dbc.Container([
    html.H1("Mini Dashboard 1: Dropdown Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Product:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='dropdown-product',
                options=[{'label': p, 'value': p} for p in df['Product'].unique()],
                value='Laptop',
                clearable=False
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='dropdown-graph')
        ])
    ])
], fluid=True)

# Callback: When dropdown changes, update the graph
@app1.callback(
    Output('dropdown-graph', 'figure'),
    Input('dropdown-product', 'value')
)
def update_dropdown_graph(selected_product):
    # Filter data for selected product
    filtered_df = df[df['Product'] == selected_product]

    # Group by month and sum sales
    monthly_sales = filtered_df.groupby('Month')['Sales'].sum().reset_index()

    # Create figure
    fig = px.line(
        monthly_sales,
        x='Month',
        y='Sales',
        title=f'Monthly Sales for {selected_product}',
        markers=True
    )
    fig.update_layout(height=400)
    return fig

print("✅ Mini Dashboard 1 (Dropdown) created!")
print("\nTo run: app1.run(debug=True, port=8070)")

✅ Mini Dashboard 1 (Dropdown) created!

To run: app1.run(debug=True, port=8070)


In [ ]:
app1.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 2: Slider Input

## Concept:
A **slider** allows users to select a **numeric value** from a range. Perfect for filtering by quantity, price, or time periods.

## Use Cases:
- Filter by minimum sales amount
- Filter by quantity range
- Select a year

## Callback Flow:
```
User moves slider → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 2: Slider
app2 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app2.layout = dbc.Container([
    html.H1("Mini Dashboard 2: Slider Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select Minimum Sales Amount:", style={'fontWeight': 'bold'}),
            dcc.Slider(
                id='slider-sales',
                min=df['Sales'].min(),
                max=df['Sales'].max(),
                step=100,
                value=1000,
                marks={i: f'${i}' for i in range(0, int(df['Sales'].max())+1, 1000)},
                tooltip={"placement": "top", "always_visible": True}
            )
        ], md=8)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='slider-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='slider-graph')
        ])
    ])
], fluid=True)

# Callback: When slider changes, update the graph
@app2.callback(
    [Output('slider-graph', 'figure'),
     Output('slider-info', 'children')],
    Input('slider-sales', 'value')
)
def update_slider_graph(min_sales):
    # Filter data for sales >= min_sales
    filtered_df = df[df['Sales'] >= min_sales]

    # Group by product and sum sales
    product_sales = filtered_df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.bar(
        product_sales,
        x='Product',
        y='Sales',
        title=f'Sales by Product (Minimum: ${min_sales})',
        color='Sales',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=400)

    # Info text
    info_text = f"Showing {len(filtered_df)} transactions with sales >= ${min_sales}"

    return fig, info_text

print("✅ Mini Dashboard 2 (Slider) created!")
print("\nTo run: app2.run(debug=True, port=8071)")

✅ Mini Dashboard 2 (Slider) created!

To run: app2.run(debug=True, port=8071)


In [ ]:
app2.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 3: Range Slider Input

## Concept:
A **range slider** allows users to select a **range of values** (min and max). Perfect for filtering by ranges.

## Use Cases:
- Filter by sales range
- Filter by quantity range
- Select a date range

## Callback Flow:
```
User adjusts range → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 3: Range Slider
app3 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app3.layout = dbc.Container([
    html.H1("Mini Dashboard 3: Range Slider Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select Sales Range:", style={'fontWeight': 'bold'}),
            dcc.RangeSlider(
                id='range-slider-sales',
                min=df['Sales'].min(),
                max=df['Sales'].max(),
                step=100,
                value=[1000, 3000],
                marks={i: f'${i}' for i in range(0, int(df['Sales'].max())+1, 1000)},
                tooltip={"placement": "bottom", "always_visible": True}
            )
        ], md=8)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='range-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='range-graph')
        ])
    ])
], fluid=True)

# Callback: When range slider changes, update the graph
@app3.callback(
    [Output('range-graph', 'figure'),
     Output('range-info', 'children')],
    Input('range-slider-sales', 'value')
)
def update_range_graph(sales_range):
    min_sales, max_sales = sales_range

    # Filter data for sales within range
    filtered_df = df[(df['Sales'] >= min_sales) & (df['Sales'] <= max_sales)]

    # Group by region and sum sales
    region_sales = filtered_df.groupby('Region')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.bar(
        region_sales,
        x='Region',
        y='Sales',
        title=f'Sales by Region (${min_sales} - ${max_sales})',
        color='Sales',
        color_continuous_scale='Blues'
    )
    fig.update_layout(height=400)

    # Info text
    info_text = f"Showing {len(filtered_df)} transactions with sales between ${min_sales} and ${max_sales}"

    return fig, info_text

print("✅ Mini Dashboard 3 (Range Slider) created!")
print("\nTo run: app3.run(debug=True, port=8072)")

✅ Mini Dashboard 3 (Range Slider) created!

To run: app3.run(debug=True, port=8072)


In [ ]:
app3.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 4: Radio Items Input

## Concept:
**Radio items** allow users to select **one option** from a list of options (mutually exclusive). Similar to dropdown but options are visible.

## Use Cases:
- Select a view type (table, chart, etc.)
- Select a metric to display
- Select a category

## Callback Flow:
```
User clicks radio button → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 4: Radio Items
app4 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app4.layout = dbc.Container([
    html.H1("Mini Dashboard 4: Radio Items Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.RadioItems(
                id='radio-region',
                options=[{'label': f'  {r}', 'value': r} for r in df['Region'].unique()],
                value='North',
                inline=True,
                className="mb-3"
            )
        ], md=6)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='radio-graph')
        ])
    ])
], fluid=True)

# Callback: When radio button changes, update the graph
@app4.callback(
    Output('radio-graph', 'figure'),
    Input('radio-region', 'value')
)
def update_radio_graph(selected_region):
    # Filter data for selected region
    filtered_df = df[df['Region'] == selected_region]

    # Group by product and sum sales
    product_sales = filtered_df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.pie(
        product_sales,
        names='Product',
        values='Sales',
        title=f'Sales Distribution by Product - {selected_region} Region',
        hole=0.3
    )
    fig.update_layout(height=500)
    return fig

print("✅ Mini Dashboard 4 (Radio Items) created!")
print("\nTo run: app4.run(debug=True, port=8073)")

✅ Mini Dashboard 4 (Radio Items) created!

To run: app4.run(debug=True, port=8073)


In [ ]:
app4.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 5: Checklist Input

## Concept:
A **checklist** allows users to select **multiple options** (not mutually exclusive). Perfect for multi-select filtering.

## Use Cases:
- Filter by multiple products
- Filter by multiple regions
- Select multiple categories

## Callback Flow:
```
User checks/unchecks items → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 5: Checklist
app5 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app5.layout = dbc.Container([
    html.H1("Mini Dashboard 5: Checklist Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select Products:", style={'fontWeight': 'bold'}),
            dcc.Checklist(
                id='checklist-products',
                options=[{'label': f'  {p}', 'value': p} for p in df['Product'].unique()],
                value=['Laptop', 'Monitor'],
                inline=True,
                className="mb-3"
            )
        ], md=8)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='checklist-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='checklist-graph')
        ])
    ])
], fluid=True)

# Callback: When checklist changes, update the graph
@app5.callback(
    [Output('checklist-graph', 'figure'),
     Output('checklist-info', 'children')],
    Input('checklist-products', 'value')
)
def update_checklist_graph(selected_products):
    # Handle empty selection
    if not selected_products:
        selected_products = df['Product'].unique().tolist()

    # Filter data for selected products
    filtered_df = df[df['Product'].isin(selected_products)]

    # Group by product and month
    product_monthly = filtered_df.groupby(['Month', 'Product'])['Sales'].sum().reset_index()

    # Create figure
    fig = px.line(
        product_monthly,
        x='Month',
        y='Sales',
        color='Product',
        title=f'Monthly Sales for Selected Products',
        markers=True
    )
    fig.update_layout(height=400)

    # Info text
    info_text = f"Showing data for: {', '.join(selected_products)}"

    return fig, info_text

print("✅ Mini Dashboard 5 (Checklist) created!")
print("\nTo run: app5.run(debug=True, port=8074)")

✅ Mini Dashboard 5 (Checklist) created!

To run: app5.run(debug=True, port=8074)


In [ ]:
app5.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 6: Text Input

## Concept:
A **text input** allows users to type text to search or filter data. Perfect for searching by name or keyword.

## Use Cases:
- Search for a product name
- Filter by keyword
- Enter a minimum value

## Callback Flow:
```
User types text → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 6: Text Input
app6 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app6.layout = dbc.Container([
    html.H1("Mini Dashboard 6: Text Input", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Search for a Product:", style={'fontWeight': 'bold'}),
            dcc.Input(
                id='text-input-product',
                type='text',
                placeholder='Type a product name...',
                value='',
                style={
                    'width': '100%',
                    'padding': '10px',
                    'fontSize': '16px',
                    'borderRadius': '5px',
                    'border': '1px solid #ddd'
                }
            )
        ], md=6)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='text-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='text-graph')
        ])
    ])
], fluid=True)

# Callback: When text input changes, update the graph
@app6.callback(
    [Output('text-graph', 'figure'),
     Output('text-info', 'children')],
    Input('text-input-product', 'value')
)
def update_text_graph(search_text):
    # Filter data based on text input (case-insensitive)
    if search_text:
        filtered_df = df[df['Product'].str.contains(search_text, case=False, na=False)]
    else:
        filtered_df = df

    # Group by region and sum sales
    region_sales = filtered_df.groupby('Region')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.bar(
        region_sales,
        x='Region',
        y='Sales',
        title=f'Sales by Region {f"(Search: {search_text})" if search_text else "(All Products)"}',
        color='Sales',
        color_continuous_scale='Greens'
    )
    fig.update_layout(height=400)

    # Info text
    if search_text:
        info_text = f"Found {len(filtered_df)} transactions for products containing '{search_text}'"
    else:
        info_text = f"Showing all {len(filtered_df)} transactions"

    return fig, info_text

print("✅ Mini Dashboard 6 (Text Input) created!")
print("\nTo run: app6.run(debug=True, port=8075)")

✅ Mini Dashboard 6 (Text Input) created!

To run: app6.run(debug=True, port=8075)


In [ ]:
app6.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 7: Date Picker

## Concept:
A **date picker** allows users to select a **single date**. Perfect for filtering by a specific date.

## Use Cases:
- Filter by a specific date
- Select a date to analyze
- Choose a reference date

## Callback Flow:
```
User selects a date → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 7: Date Picker
app7 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app7.layout = dbc.Container([
    html.H1("Mini Dashboard 7: Date Picker", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Date:", style={'fontWeight': 'bold'}),
            dcc.DatePickerSingle(
                id='date-picker',
                date=df['Date'].max(),
                display_format='YYYY-MM-DD',
                style={'width': '100%'}
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='date-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='date-graph')
        ])
    ])
], fluid=True)

# Callback: When date picker changes, update the graph
@app7.callback(
    [Output('date-graph', 'figure'),
     Output('date-info', 'children')],
    Input('date-picker', 'date')
)
def update_date_graph(selected_date):
    # Convert to datetime
    selected_date = pd.to_datetime(selected_date)

    # Filter data for selected date
    filtered_df = df[df['Date'].dt.date == selected_date.date()]

    # Group by product and sum sales
    product_sales = filtered_df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.bar(
        product_sales,
        x='Product',
        y='Sales',
        title=f'Sales by Product on {selected_date.strftime("%B %d, %Y")}',
        color='Sales',
        color_continuous_scale='Oranges'
    )
    fig.update_layout(height=400)

    # Info text
    info_text = f"Showing {len(filtered_df)} transactions on {selected_date.strftime('%B %d, %Y')}"

    return fig, info_text

print("✅ Mini Dashboard 7 (Date Picker) created!")
print("\nTo run: app7.run(debug=True, port=8076)")

✅ Mini Dashboard 7 (Date Picker) created!

To run: app7.run(debug=True, port=8076)


In [ ]:
app7.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 8: Date Range Picker

## Concept:
A **date range picker** allows users to select a **range of dates**. Perfect for analyzing trends over time periods.

## Use Cases:
- Filter by date range
- Analyze trends for a specific period
- Compare different time periods

## Callback Flow:
```
User selects date range → Callback triggered → Graph updates
```

In [ ]:
# Mini Dashboard 8: Date Range Picker
app8 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app8.layout = dbc.Container([
    html.H1("Mini Dashboard 8: Date Range Picker", className="mb-4 mt-4"),

    dbc.Row([
        dbc.Col([
            html.Label("Select Date Range:", style={'fontWeight': 'bold'}),
            dcc.DatePickerRange(
                id='date-range-picker',
                start_date=df['Date'].min(),
                end_date=df['Date'].max(),
                display_format='YYYY-MM-DD',
                style={'width': '100%'}
            )
        ], md=8)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='date-range-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='date-range-graph')
        ])
    ])
], fluid=True)

# Callback: When date range picker changes, update the graph
@app8.callback(
    [Output('date-range-graph', 'figure'),
     Output('date-range-info', 'children')],
    [Input('date-range-picker', 'start_date'),
     Input('date-range-picker', 'end_date')]
)
def update_date_range_graph(start_date, end_date):
    # Convert to datetime
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Filter data for date range
    filtered_df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

    # Group by month and sum sales
    monthly_sales = filtered_df.groupby('Month')['Sales'].sum().reset_index()

    # Create figure
    fig = px.line(
        monthly_sales,
        x='Month',
        y='Sales',
        title=f'Monthly Sales ({start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")})',
        markers=True
    )
    fig.update_layout(height=400)

    # Info text
    days = (end_date - start_date).days
    info_text = f"Showing {len(filtered_df)} transactions over {days} days"

    return fig, info_text

print("✅ Mini Dashboard 8 (Date Range Picker) created!")
print("\nTo run: app8.run(debug=True, port=8077)")

✅ Mini Dashboard 8 (Date Range Picker) created!

To run: app8.run(debug=True, port=8077)


In [ ]:
app8.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 9: Button Input

## Concept:
A **button** allows users to trigger actions. Combined with **State**, it can be used to submit form data.

## Use Cases:
- Submit filters
- Trigger data refresh
- Reset filters
- Download data

## Callback Flow:
```
User clicks button → Callback triggered → Action performed
```

## Key Concept: State vs Input
- **Input**: Triggers callback when value changes
- **State**: Provides value to callback but doesn't trigger it

In [ ]:
# Mini Dashboard 9: Button
app9 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app9.layout = dbc.Container([
    html.H1("Mini Dashboard 9: Button Input", className="mb-4 mt-4"),
    dcc.Store(id='click-store', data={'North': 0, 'South': 0, 'East': 0, 'West': 0}),

    dbc.Row([
        dbc.Col([
            html.Label("Select a Region:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='button-region-dropdown',
                options=[{'label': r, 'value': r} for r in df['Region'].unique()],
                value='North'
            )
        ], md=4),

        dbc.Col([
            html.Label("Action:", style={'fontWeight': 'bold'}),
            dbc.Button(
                "Generate Report",
                id='generate-button',
                color='primary',
                className="w-100"
            )
        ], md=4)
    ], className="mb-4"),

    dbc.Row([
        dbc.Col([
            html.H5(id='button-info', className="mb-3")
        ])
    ]),

    dbc.Row([
        dbc.Col([
            dcc.Graph(id='button-graph')
        ])
    ])
], fluid=True)

# Callback: When button is clicked, generate report
# Note: Using State - dropdown value doesn't trigger callback, button click does
@app9.callback(
    [Output('button-graph', 'figure'),
     Output('button-info', 'children'),
    Output('click-store', 'data')],
    Input('generate-button', 'n_clicks'),
    State('button-region-dropdown', 'value'),
    State('click-store', 'data'),
    prevent_initial_call=True
)
def update_button_graph(n_clicks, selected_region, store_data):
    if not n_clicks:
        raise PreventUpdate
  # Increment click count for this region
    store_data[selected_region] = store_data.get(selected_region, 0) + 1
    region_clicks = store_data[selected_region]

    # Filter data for selected region
    filtered_df = df[df['Region'] == selected_region]

    # Group by product and sum sales
    product_sales = filtered_df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)

    # Create figure
    fig = px.bar(
        product_sales,
        x='Product',
        y='Sales',
        title=f'Sales Report for {selected_region} Region',
        color='Sales',
        color_continuous_scale='Reds'
    )
    fig.update_layout(height=400)

    # Info text
    info_text = f"Report generated {region_clicks} time(s) for {selected_region} region"

    return fig, info_text, store_data

print("✅ Mini Dashboard 9 (Button) created!")
print("\nTo run: app9.run(debug=True, port=8078)")

✅ Mini Dashboard 9 (Button) created!

To run: app9.run(debug=True, port=8078)


In [ ]:
app9.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Mini Dashboard 10: Tabs

## Concept:
**Tabs** allow users to switch between different views or sections. Each tab can contain different content.

## Use Cases:
- Switch between different reports
- Show different views of the same data
- Organize content into sections

## Callback Flow:
```
User clicks tab → Callback triggered → Content updates
```

In [ ]:
# Mini Dashboard 10: Tabs
app10 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app10.layout = dbc.Container([
    html.H1("Mini Dashboard 10: Tabs", className="mb-4 mt-4"),

    dcc.Tabs(id='tabs', value='tab-1', children=[
        dcc.Tab(label='Sales by Product', value='tab-1', children=[
            dbc.Container([
                html.Br(),
                dcc.Graph(id='tab1-graph')
            ])
        ]),

        dcc.Tab(label='Sales by Region', value='tab-2', children=[
            dbc.Container([
                html.Br(),
                dcc.Graph(id='tab2-graph')
            ])
        ]),

        dcc.Tab(label='Sales by Category', value='tab-3', children=[
            dbc.Container([
                html.Br(),
                dcc.Graph(id='tab3-graph')
            ])
        ])
    ])
], fluid=True)

# Callback for Tab 1: Sales by Product
@app10.callback(
    Output('tab1-graph', 'figure'),
    Input('tabs', 'value')
)
def update_tab1(active_tab):
    product_sales = df.groupby('Product')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)
    fig = px.bar(
        product_sales,
        x='Product',
        y='Sales',
        title='Total Sales by Product',
        color='Sales',
        color_continuous_scale='Blues'
    )
    fig.update_layout(height=500)
    return fig

# Callback for Tab 2: Sales by Region
@app10.callback(
    Output('tab2-graph', 'figure'),
    Input('tabs', 'value')
)
def update_tab2(active_tab):
    region_sales = df.groupby('Region')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)
    fig = px.pie(
        region_sales,
        names='Region',
        values='Sales',
        title='Sales Distribution by Region',
        hole=0.3
    )
    fig.update_layout(height=500)
    return fig

# Callback for Tab 3: Sales by Category
@app10.callback(
    Output('tab3-graph', 'figure'),
    Input('tabs', 'value')
)
def update_tab3(active_tab):
    category_sales = df.groupby('Category')['Sales'].sum().reset_index()
    fig = px.bar(
        category_sales,
        x='Category',
        y='Sales',
        title='Total Sales by Category',
        color='Sales',
        color_continuous_scale='Greens'
    )
    fig.update_layout(height=500)
    return fig

print("✅ Mini Dashboard 10 (Tabs) created!")
print("\nTo run: app10.run(debug=True, port=8079)")

✅ Mini Dashboard 10 (Tabs) created!

To run: app10.run(debug=True, port=8079)


In [ ]:
app10.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>

---

# Summary: Input Types and Callbacks

## Quick Reference Table

| Input Type | Component | Use Case | Example |
|-----------|-----------|----------|----------|
| **Dropdown** | `dcc.Dropdown` | Select one option | Product selection |
| **Slider** | `dcc.Slider` | Select numeric value | Minimum sales amount |
| **Range Slider** | `dcc.RangeSlider` | Select range | Sales range |
| **Radio Items** | `dcc.RadioItems` | Select one (visible) | Region selection |
| **Checklist** | `dcc.Checklist` | Select multiple | Multiple products |
| **Text Input** | `dcc.Input` | Type text | Search/filter |
| **Date Picker** | `dcc.DatePickerSingle` | Select one date | Specific date analysis |
| **Date Range** | `dcc.DatePickerRange` | Select date range | Period analysis |
| **Button** | `dbc.Button` | Trigger action | Submit/refresh |
| **Tabs** | `dcc.Tabs` | Switch views | Different reports |

---

## Callback Patterns

### Pattern 1: Single Input → Single Output
```python
@app.callback(
    Output('graph', 'figure'),
    Input('dropdown', 'value')
)
def update_graph(selected_value):
    # Update logic
    return figure
```

### Pattern 2: Single Input → Multiple Outputs
```python
@app.callback(
    [Output('graph', 'figure'),
     Output('info', 'children')],
    Input('slider', 'value')
)
def update_multiple(selected_value):
    return figure, info_text
```

### Pattern 3: Multiple Inputs → Single Output
```python
@app.callback(
    Output('graph', 'figure'),
    [Input('dropdown', 'value'),
     Input('slider', 'value')]
)
def update_graph(dropdown_val, slider_val):
    # Both inputs trigger callback
    return figure
```

### Pattern 4: Input + State → Output (Button Pattern)
```python
@app.callback(
    Output('graph', 'figure'),
    Input('button', 'n_clicks'),
    State('dropdown', 'value')
)
def update_on_button(n_clicks, dropdown_val):
    # Only button click triggers, dropdown value is just read
    return figure
```

---

## Key Concepts

### Input vs State
- **Input**: Triggers callback when value changes
- **State**: Provides value but doesn't trigger callback
- Use **State** with **Button** for form submission pattern

### Callback Execution
1. User interacts with input component
2. Input value changes
3. Callback is triggered
4. Function executes
5. Output component updates

---

## Best Practices

1. **Keep callbacks simple** - One callback per logical operation
2. **Use clear variable names** - Makes code readable
3. **Add error handling** - Handle empty selections
4. **Provide feedback** - Show info text about current filters
5. **Test edge cases** - Empty selections, extreme values, etc.
6. **Use State for buttons** - Don't trigger on every input change
7. **Document callbacks** - Explain what each callback does

---





## Resources

- [Dash Callback Documentation](https://dash.plotly.com/basic-callbacks)
- [Dash Core Components](https://dash.plotly.com/dash-core-components)
- [Dash Bootstrap Components](https://dash-bootstrap-components.opensource.faculty.ai/)
- [Plotly Chart Types](https://plotly.com/python/)